# Advanced Orbital Analysis: 5-Phase Systems and Power Flow

This notebook extends the 3-phase orbital framework to:
1. **d-orbitals as 5-phase systems** - higher-order symmetrical components
2. **Time-dependent superpositions** - unbalanced systems and beating modes
3. **Sequence power flow** - cross-terms between different m values

## Mathematical Framework

For orbital l, there are (2l+1) magnetic substates m = -l, ..., +l:
- l=0 (s): 1 component (single-phase)
- l=1 (p): 3 components (3-phase)
- l=2 (d): 5 components (5-phase)
- l=3 (f): 7 components (7-phase)

These form **multipole expansions** - just like how multi-phase systems create
higher-order rotating fields!

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from mpl_toolkits.mplot3d import Axes3D
from scipy.special import sph_harm_y, genlaguerre, factorial
from matplotlib import cm
from matplotlib.colors import Normalize
from matplotlib.animation import FuncAnimation
from IPython.display import HTML
import warnings
warnings.filterwarnings('ignore')

# Set high DPI for better quality plots
plt.rcParams['figure.dpi'] = 100

In [ ]:
def R_nl(r, n, l, a0=1.0):
    """Hydrogen radial wavefunction R_{nl}(r)."""
    rho = 2.0 * r / (n * a0)
    L = genlaguerre(n - l - 1, 2*l + 1)(rho)
    pref = (2.0/(n*a0))**3
    norm = np.sqrt(pref * factorial(n - l - 1) / (2*n * factorial(n + l)))
    return norm * np.exp(-rho/2) * rho**l * L

def Y_lm(theta, phi, l, m):
    """Spherical harmonic Y_l^m(theta, phi)."""
    return sph_harm_y(m, l, phi, theta)

def psi_nlm(r, theta, phi, n, l, m, a0=1.0):
    """Full spatial wavefunction ψ_{nlm}(r,θ,φ)."""
    return R_nl(r, n, l, a0=a0) * Y_lm(theta, phi, l, m)

def psi_nlm_time(r, theta, phi, t, n, l, m, a0=1.0, hbar=1.0):
    """Time-dependent wavefunction ψ(r,θ,φ,t) = ψ(r,θ,φ)·exp(-iEt/ℏ)."""
    E_n = -13.6 / n**2  # Energy in eV
    omega = E_n / hbar  # Angular frequency
    spatial = psi_nlm(r, theta, phi, n, l, m, a0=a0)
    return spatial * np.exp(-1j * omega * t)

def density_nlm(r, theta, phi, n, l, m, a0=1.0):
    """Probability density |ψ|²."""
    psi = psi_nlm(r, theta, phi, n, l, m, a0=a0)
    return np.abs(psi)**2

## 1. d-Orbitals as a 5-Phase System

The five d-orbitals (l=2) form a complete set with m = -2, -1, 0, +1, +2.

### Sequence Component Interpretation:
- **m = ±2**: Double-frequency sequence (2nd harmonic, rotates as e^(±2iφ))
- **m = ±1**: Fundamental sequence (rotates as e^(±iφ))
- **m = 0**: Zero sequence (no rotation)

### 5-Phase Power Analogy:
In a 5-phase system with phases at 0°, 72°, 144°, 216°, 288°:
- Positive sequence rotates with fundamental frequency
- **2nd harmonic** appears naturally (like m=±2 in d-orbitals)
- Symmetrical component decomposition has 5 components

The d-orbitals capture this richer harmonic structure!

In [ ]:
# Visualize all 5 d-orbital sequence components
phi_vals = np.linspace(0, 2*np.pi, 300)
theta_fixed = np.pi/2  # Equatorial plane

fig, axes = plt.subplots(5, 1, figsize=(14, 20))
fig.suptitle('d-Orbitals: 5-Phase System with Harmonic Sequences', fontsize=16, fontweight='bold')

m_values = [2, 1, 0, -1, -2]
labels = [
    '+2 sequence (2nd harmonic, CCW)',
    '+1 sequence (fundamental, CCW)',
    '0 sequence (stationary)',
    '-1 sequence (fundamental, CW)',
    '-2 sequence (2nd harmonic, CW)'
]
colors = ['red', 'orange', 'green', 'blue', 'purple']

for idx, (m, label, color) in enumerate(zip(m_values, labels, colors)):
    ax = axes[idx]
    
    Y = Y_lm(theta_fixed, phi_vals, l=2, m=m)
    magnitude = np.abs(Y)**2
    
    # Plot magnitude
    ax.plot(phi_vals, magnitude, linewidth=3, color=color, label=f'|Y_2^{{{m}}}|²')
    ax.fill_between(phi_vals, 0, magnitude, alpha=0.3, color=color)
    
    # Also plot real and imaginary parts to show rotation
    ax2 = ax.twinx()
    ax2.plot(phi_vals, np.real(Y), '--', linewidth=1.5, alpha=0.7, color='darkred', label='Re(Y)')
    ax2.plot(phi_vals, np.imag(Y), ':', linewidth=1.5, alpha=0.7, color='darkblue', label='Im(Y)')
    ax2.set_ylabel('Re/Im components', fontsize=10)
    ax2.legend(loc='upper right', fontsize=8)
    
    ax.set_title(f'Y_2^{{{m}}}: {label}', fontweight='bold', fontsize=12)
    ax.set_xlabel('φ (radians)', fontsize=11)
    ax.set_ylabel('Probability Density', fontsize=11)
    ax.legend(loc='upper left', fontsize=10)
    ax.grid(True, alpha=0.3)
    ax.axhline(0, color='k', linewidth=0.5)
    
    # Add harmonic annotation
    if abs(m) == 2:
        harmonic_text = f'Rotates as e^{{±2iφ}} (2nd harmonic)'
    elif abs(m) == 1:
        harmonic_text = f'Rotates as e^{{±iφ}} (fundamental)'
    else:
        harmonic_text = 'No φ dependence (DC/zero sequence)'
    
    ax.text(0.02, 0.95, harmonic_text, transform=ax.transAxes,
            fontsize=9, verticalalignment='top',
            bbox=dict(boxstyle='round', facecolor='wheat', alpha=0.8))

plt.tight_layout()
plt.show()

print("\n=== 5-Phase Power System Analogy ===")
print("d-orbitals (l=2) have 5 components, analogous to 5-phase power.")
print("Notice the 2nd harmonic components (m=±2) - these appear naturally")
print("in 5-phase systems and represent higher-order rotating fields!")
print("\nPhase angles for balanced 5-phase: 0°, 72°, 144°, 216°, 288°")
print("Symmetrical components: 0, ±1, ±2 (zero, fundamental, 2nd harmonic)")

In [ ]:
# Visualize d-orbitals in 2D slices to see spatial patterns
fig, axes = plt.subplots(2, 3, figsize=(16, 11))
fig.suptitle('d-Orbital Spatial Patterns: 5-Phase Harmonic Structure', fontsize=14, fontweight='bold')

extent = 15
x = np.linspace(-extent, extent, 200)
y = np.linspace(-extent, extent, 200)
X, Y = np.meshgrid(x, y)
Z_plane = 0

# Convert to spherical
R_grid = np.sqrt(X**2 + Y**2 + Z_plane**2)
THETA_grid = np.arccos(Z_plane / (R_grid + 1e-10))
PHI_grid = np.arctan2(Y, X)

m_values = [-2, -1, 0, 1, 2]

for idx, m in enumerate(m_values):
    ax = axes[idx // 3, idx % 3]
    
    psi = psi_nlm(R_grid, THETA_grid, PHI_grid, n=3, l=2, m=m)
    density = np.abs(psi)**2
    
    # Use symmetric colormap centered at 0 for the wavefunction
    im = ax.contourf(X, Y, density, levels=30, cmap='viridis')
    ax.contour(X, Y, density, levels=10, colors='black', linewidths=0.5, alpha=0.3)
    
    ax.set_xlabel('x (Bohr radii)')
    ax.set_ylabel('y (Bohr radii)' if idx % 3 == 0 else '')
    ax.set_title(f'd-orbital: m = {m:+d}\n({"2nd harmonic" if abs(m)==2 else "fundamental" if abs(m)==1 else "zero seq"})',
                 fontweight='bold')
    ax.set_aspect('equal')
    ax.grid(True, alpha=0.2)
    
    plt.colorbar(im, ax=ax, label='|ψ|²')
    
    # Add lobes count
    lobes = 2 * abs(m) if m != 0 else 1
    ax.text(0.02, 0.98, f'{lobes} lobe(s)', transform=ax.transAxes,
            fontsize=9, verticalalignment='top',
            bbox=dict(boxstyle='round', facecolor='lightblue', alpha=0.7))

# Remove the 6th subplot
fig.delaxes(axes[1, 2])

plt.tight_layout()
plt.show()

print("\n=== Pattern Recognition ===")
print("Notice how the number of lobes increases with |m|:")
print("  m = 0: Single lobe (like DC component)")
print("  m = ±1: 2 lobes (fundamental dipole)")
print("  m = ±2: 4 lobes (2nd harmonic quadrupole)")
print("\nThis is exactly like harmonic content in multi-phase systems!")

## 2. Time-Dependent Superpositions: Unbalanced Systems and Beating

When we superpose orbitals with different energies, we get **time-dependent behavior** -
exactly like beating in unbalanced multi-phase systems!

### Key Concept:
A general state: ψ(t) = c₁·ψ₁·e^(-iE₁t/ℏ) + c₂·ψ₂·e^(-iE₂t/ℏ)

This creates **beating** at frequency Δω = (E₂-E₁)/ℏ

### Power System Analogy:
- **Balanced 3-phase**: All phases at same frequency → stationary rotating field
- **Unbalanced**: Different harmonics → beating, pulsating fields
- **Quantum superposition**: Different energy levels → oscillating probability density

### Example: 2s + 2p Superposition
The 2s and 2p orbitals have the same energy (n=2), so adding them creates a
**static dipole** (no beating). But 1s + 2p creates oscillations at the
Lyman-alpha frequency!

In [ ]:
def superposition_state(r, theta, phi, t, coeffs, states, hbar=1.0):
    """
    Create superposition of multiple states.
    
    coeffs: list of complex coefficients [c1, c2, ...]
    states: list of (n, l, m) tuples
    """
    psi_total = np.zeros_like(r, dtype=complex)
    
    for c, (n, l, m) in zip(coeffs, states):
        psi_total += c * psi_nlm_time(r, theta, phi, t, n, l, m, hbar=hbar)
    
    return psi_total

def expectation_dipole_moment(r, theta, phi, psi):
    """
    Calculate expectation value of dipole moment <ψ|z|ψ>.
    This oscillates in time for superpositions!
    """
    z = r * np.cos(theta)
    vol_element = r**2 * np.sin(theta)
    
    # Approximate integral via sampling
    integrand = np.conj(psi) * z * psi * vol_element
    return np.sum(np.real(integrand))

In [ ]:
# Example 1: 1s + 2p superposition (creates beating/oscillating dipole)
print("=== Example 1: 1s + 2p Superposition (Lyman-alpha transition) ===")
print("Energy difference: ΔE = E₂ - E₁ = -13.6/4 - (-13.6/1) = 10.2 eV")
print("Beat frequency: ω = ΔE/ℏ ≈ 1.55 × 10¹⁶ rad/s")
print("Period: T = 2π/ω ≈ 4.05 × 10⁻¹⁶ s (femtosecond scale)")
print("\nThis is like two AC sources at different frequencies beating!")

# Time evolution
n_times = 100
t_vals = np.linspace(0, 2*np.pi, n_times)  # Normalized time units

# Setup grid (1D slice along z-axis)
z_vals = np.linspace(-20, 20, 200)
r_1d = np.abs(z_vals)
theta_1d = np.where(z_vals >= 0, 0, np.pi)
phi_1d = np.zeros_like(z_vals)

# Equal superposition of 1s and 2pz
c1 = 1/np.sqrt(2)
c2 = 1/np.sqrt(2)
states = [(1, 0, 0), (2, 1, 0)]  # 1s and 2p_z

# Calculate density evolution
densities_1s2p = []
dipole_moments = []

for t in t_vals:
    psi = superposition_state(r_1d, theta_1d, phi_1d, t, [c1, c2], states)
    density = np.abs(psi)**2
    densities_1s2p.append(density)
    
    # Calculate dipole moment
    dipole = expectation_dipole_moment(r_1d, theta_1d, phi_1d, psi)
    dipole_moments.append(dipole)

densities_1s2p = np.array(densities_1s2p)

# Plot density evolution
fig, axes = plt.subplots(2, 1, figsize=(14, 10))

# Density as function of position and time
ax = axes[0]
im = ax.contourf(z_vals, t_vals, densities_1s2p, levels=50, cmap='hot')
ax.set_xlabel('z position (Bohr radii)', fontsize=12)
ax.set_ylabel('Time (normalized)', fontsize=12)
ax.set_title('1s + 2p Superposition: Beating Density Pattern', fontweight='bold', fontsize=13)
plt.colorbar(im, ax=ax, label='Probability Density |ψ|²')

# Add annotations
ax.text(0.02, 0.98, 'Notice the oscillating pattern!\nLike voltage envelope in AM modulation',
        transform=ax.transAxes, fontsize=10, verticalalignment='top',
        bbox=dict(boxstyle='round', facecolor='lightblue', alpha=0.9))

# Dipole moment oscillation
ax = axes[1]
ax.plot(t_vals, dipole_moments, 'b-', linewidth=2.5, label='<z> dipole moment')
ax.axhline(0, color='k', linewidth=0.5, linestyle='--')
ax.set_xlabel('Time (normalized)', fontsize=12)
ax.set_ylabel('Dipole Moment <z>', fontsize=12)
ax.set_title('Oscillating Dipole Moment (analogous to reactive power oscillation)', fontweight='bold', fontsize=13)
ax.grid(True, alpha=0.3)
ax.legend(fontsize=11)

# Add power analogy
ax.text(0.02, 0.98, 'Like Q(t) in unbalanced 3-phase system\nReactive power oscillates at beat frequency',
        transform=ax.transAxes, fontsize=10, verticalalignment='top',
        bbox=dict(boxstyle='round', facecolor='wheat', alpha=0.9))

plt.tight_layout()
plt.show()

print("\n=== Physical Interpretation ===")
print("The electron density 'sloshes' back and forth (dipole oscillation)")
print("This creates electromagnetic radiation at the beat frequency!")
print("Exactly analogous to power flowing in/out of reactive elements in unbalanced AC system.")

In [ ]:
# Example 2: 2s + 2p superposition (same energy, no beating)
print("\n=== Example 2: 2s + 2p Superposition (Degenerate States) ===")
print("Energy difference: ΔE = 0 (both have n=2)")
print("No beating! Creates a static polarized orbital.")
print("Like balanced 3-phase at same frequency → steady rotating field.")

# Equal superposition of 2s and 2pz
states_2s2p = [(2, 0, 0), (2, 1, 0)]  # 2s and 2p_z

# This should be time-independent (up to overall phase)
densities_2s2p = []
for t in np.linspace(0, 2*np.pi, 5):  # Just a few times
    psi = superposition_state(r_1d, theta_1d, phi_1d, t, [c1, c2], states_2s2p)
    density = np.abs(psi)**2
    densities_2s2p.append(density)

# Plot to verify time-independence
fig, ax = plt.subplots(figsize=(12, 6))
for idx, density in enumerate(densities_2s2p):
    ax.plot(z_vals, density, linewidth=2, alpha=0.7, label=f't = {idx}')

ax.set_xlabel('z position (Bohr radii)', fontsize=12)
ax.set_ylabel('Probability Density |ψ|²', fontsize=12)
ax.set_title('2s + 2p Superposition: Time-Independent (Degenerate)', fontweight='bold', fontsize=13)
ax.legend(fontsize=10)
ax.grid(True, alpha=0.3)

ax.text(0.02, 0.98, 'All curves overlap!\nLike balanced 3-phase: steady-state pattern',
        transform=ax.transAxes, fontsize=10, verticalalignment='top',
        bbox=dict(boxstyle='round', facecolor='lightgreen', alpha=0.9))

plt.tight_layout()
plt.show()

print("\nNotice: All time snapshots overlap perfectly!")
print("The density pattern is static - just a polarized orbital pointing along +z.")

## 3. Sequence Power Flow: Cross-Terms Between m Values

In power systems, **sequence power flow** describes how positive, negative, and zero
sequence components interact. Cross-terms between sequences can indicate:
- Unbalanced loads
- Harmonic distortion
- Inter-sequence power transfer

### Quantum Analog:
When we have superpositions of different m values, the density involves **cross-terms**:

|ψ₁ + ψ₂|² = |ψ₁|² + |ψ₂|² + 2·Re(ψ₁*·ψ₂)

The cross-term 2·Re(ψ₁*·ψ₂) is analogous to **inter-sequence power flow**!

### Key Insight:
- Diagonal terms |ψₘ|² → "self power" in each sequence
- Off-diagonal terms ψₘ*·ψₘ' → "cross-sequence power flow"
- These create interference patterns (constructive/destructive)

### Selection Rules:
Just like in power systems where certain sequence combinations don't interact,
quantum mechanics has **selection rules** determining which m values couple!

In [ ]:
def analyze_sequence_power_flow(r, theta, phi, n, l, m1, m2):
    """
    Analyze 'power flow' between two sequence components (m values).
    
    Returns:
    - Individual densities |ψ₁|², |ψ₂|²
    - Total density |ψ₁ + ψ₂|²
    - Cross-term (interference): 2·Re(ψ₁*·ψ₂)
    """
    psi1 = psi_nlm(r, theta, phi, n, l, m1)
    psi2 = psi_nlm(r, theta, phi, n, l, m2)
    
    density1 = np.abs(psi1)**2
    density2 = np.abs(psi2)**2
    
    # Superposition
    psi_super = (psi1 + psi2) / np.sqrt(2)  # Normalized
    density_super = np.abs(psi_super)**2
    
    # Cross-term (interference)
    cross_term = 2 * np.real(psi1 * np.conj(psi2)) / 2  # Factor for normalization
    
    return density1, density2, density_super, cross_term

print("=== Sequence Power Flow Analysis ===")
print("We'll examine cross-terms between different m values (sequence components).")
print("This is analogous to analyzing power flow between positive/negative/zero sequences.")

In [ ]:
# Example 1: p-orbitals (l=1): m=+1 and m=-1 (positive/negative sequence)
print("\n=== p-Orbitals: Positive (+1) and Negative (-1) Sequence Interaction ===")

extent = 12
x = np.linspace(-extent, extent, 250)
y = np.linspace(-extent, extent, 250)
X, Y = np.meshgrid(x, y)
Z_plane = 0

R_grid = np.sqrt(X**2 + Y**2 + Z_plane**2)
THETA_grid = np.arccos(Z_plane / (R_grid + 1e-10))
PHI_grid = np.arctan2(Y, X)

# Analyze m=+1 and m=-1 interaction
d1, d2, d_super, cross = analyze_sequence_power_flow(
    R_grid, THETA_grid, PHI_grid, n=2, l=1, m1=1, m2=-1
)

fig, axes = plt.subplots(2, 2, figsize=(14, 14))
fig.suptitle('Sequence Power Flow: p-Orbitals (m=+1 ↔ m=-1)', fontsize=14, fontweight='bold')

# Individual densities
im1 = axes[0, 0].contourf(X, Y, d1, levels=30, cmap='Reds')
axes[0, 0].set_title('Positive Sequence: |ψ₊₁|²', fontweight='bold')
axes[0, 0].set_xlabel('x')
axes[0, 0].set_ylabel('y')
axes[0, 0].set_aspect('equal')
plt.colorbar(im1, ax=axes[0, 0])

im2 = axes[0, 1].contourf(X, Y, d2, levels=30, cmap='Blues')
axes[0, 1].set_title('Negative Sequence: |ψ₋₁|²', fontweight='bold')
axes[0, 1].set_xlabel('x')
axes[0, 1].set_ylabel('y')
axes[0, 1].set_aspect('equal')
plt.colorbar(im2, ax=axes[0, 1])

# Superposition
im3 = axes[1, 0].contourf(X, Y, d_super, levels=30, cmap='viridis')
axes[1, 0].set_title('Total: |ψ₊₁ + ψ₋₁|² / √2', fontweight='bold')
axes[1, 0].set_xlabel('x')
axes[1, 0].set_ylabel('y')
axes[1, 0].set_aspect('equal')
plt.colorbar(im3, ax=axes[1, 0])

# Cross-term (interference)
im4 = axes[1, 1].contourf(X, Y, cross, levels=30, cmap='RdBu_r')
axes[1, 1].contour(X, Y, cross, levels=[0], colors='black', linewidths=2)
axes[1, 1].set_title('Inter-Sequence Power Flow: 2·Re(ψ₊₁*·ψ₋₁)', fontweight='bold')
axes[1, 1].set_xlabel('x')
axes[1, 1].set_ylabel('y')
axes[1, 1].set_aspect('equal')
plt.colorbar(im4, ax=axes[1, 1], label='Interference term')

# Add annotations
axes[1, 1].text(0.02, 0.98, 'Red: constructive interference\nBlue: destructive interference',
                transform=axes[1, 1].transAxes, fontsize=9, verticalalignment='top',
                bbox=dict(boxstyle='round', facecolor='white', alpha=0.8))

plt.tight_layout()
plt.show()

print("\n=== Power System Interpretation ===")
print("The cross-term shows where positive and negative sequences 'exchange power'.")
print("This creates the interference pattern - regions of enhancement and cancellation.")
print("In power systems: S_pn = V_p · I_n* (cross-sequence apparent power)")

In [ ]:
# Example 2: d-orbitals (l=2): m=+2 and m=-2 (2nd harmonic sequences)
print("\n=== d-Orbitals: 2nd Harmonic Sequences (m=+2 ↔ m=-2) ===")

# Analyze m=+2 and m=-2 interaction
d1, d2, d_super, cross = analyze_sequence_power_flow(
    R_grid, THETA_grid, PHI_grid, n=3, l=2, m1=2, m2=-2
)

fig, axes = plt.subplots(2, 2, figsize=(14, 14))
fig.suptitle('Sequence Power Flow: d-Orbitals (m=+2 ↔ m=-2, 2nd Harmonic)', 
             fontsize=14, fontweight='bold')

# Individual densities
im1 = axes[0, 0].contourf(X, Y, d1, levels=30, cmap='Reds')
axes[0, 0].set_title('+2 Sequence: |ψ₊₂|² (2nd harmonic CCW)', fontweight='bold')
axes[0, 0].set_xlabel('x')
axes[0, 0].set_ylabel('y')
axes[0, 0].set_aspect('equal')
plt.colorbar(im1, ax=axes[0, 0])

im2 = axes[0, 1].contourf(X, Y, d2, levels=30, cmap='Blues')
axes[0, 1].set_title('-2 Sequence: |ψ₋₂|² (2nd harmonic CW)', fontweight='bold')
axes[0, 1].set_xlabel('x')
axes[0, 1].set_ylabel('y')
axes[0, 1].set_aspect('equal')
plt.colorbar(im2, ax=axes[0, 1])

# Superposition
im3 = axes[1, 0].contourf(X, Y, d_super, levels=30, cmap='viridis')
axes[1, 0].set_title('Total: |ψ₊₂ + ψ₋₂|² / √2', fontweight='bold')
axes[1, 0].set_xlabel('x')
axes[1, 0].set_ylabel('y')
axes[1, 0].set_aspect('equal')
plt.colorbar(im3, ax=axes[1, 0])

# Cross-term
im4 = axes[1, 1].contourf(X, Y, cross, levels=30, cmap='RdBu_r')
axes[1, 1].contour(X, Y, cross, levels=[0], colors='black', linewidths=2)
axes[1, 1].set_title('2nd Harmonic Inter-Sequence Flow: 2·Re(ψ₊₂*·ψ₋₂)', fontweight='bold')
axes[1, 1].set_xlabel('x')
axes[1, 1].set_ylabel('y')
axes[1, 1].set_aspect('equal')
plt.colorbar(im4, ax=axes[1, 1], label='Interference')

axes[1, 1].text(0.02, 0.98, 'Notice 4-fold symmetry!\n(2nd harmonic → 4 lobes)',
                transform=axes[1, 1].transAxes, fontsize=9, verticalalignment='top',
                bbox=dict(boxstyle='round', facecolor='yellow', alpha=0.8))

plt.tight_layout()
plt.show()

print("\n=== Higher Harmonic Interaction ===")
print("m=±2 components rotate twice as fast → 2nd harmonic")
print("Their interference creates 4-lobe pattern (characteristic of quadrupole)")
print("Like 2nd harmonic power flow in 5-phase systems!")

In [ ]:
# Comprehensive power flow analysis: Cross-term matrix
print("\n=== Complete Sequence Power Flow Matrix ===")
print("Computing all cross-terms for d-orbitals (l=2, m=-2 to +2)\n")

# Sample points for integration
n_samples = 5000
r_sample = np.random.exponential(3, n_samples)
theta_sample = np.arccos(2*np.random.rand(n_samples) - 1)
phi_sample = 2*np.pi*np.random.rand(n_samples)
vol_element = r_sample**2 * np.sin(theta_sample)

m_values = [-2, -1, 0, 1, 2]
n_orbitals = len(m_values)

# Compute all orbitals
orbitals = {}
for m in m_values:
    orbitals[m] = psi_nlm(r_sample, theta_sample, phi_sample, n=3, l=2, m=m)

# Cross-term matrix
power_flow_matrix = np.zeros((n_orbitals, n_orbitals))

for i, m1 in enumerate(m_values):
    for j, m2 in enumerate(m_values):
        # Calculate <ψ_m1 | ψ_m2> (approximately)
        cross_term = orbitals[m1] * np.conj(orbitals[m2]) * vol_element
        power_flow_matrix[i, j] = np.abs(np.sum(cross_term)) / n_samples

# Normalize
power_flow_matrix = power_flow_matrix / np.max(power_flow_matrix)

# Visualize
fig, ax = plt.subplots(figsize=(10, 9))
im = ax.imshow(power_flow_matrix, cmap='hot', interpolation='nearest')

# Labels
labels = [f'm={m:+d}' for m in m_values]
ax.set_xticks(range(n_orbitals))
ax.set_yticks(range(n_orbitals))
ax.set_xticklabels(labels)
ax.set_yticklabels(labels)

# Annotate values
for i in range(n_orbitals):
    for j in range(n_orbitals):
        text = ax.text(j, i, f'{power_flow_matrix[i, j]:.2f}',
                      ha="center", va="center", color="white" if power_flow_matrix[i, j] > 0.5 else "black",
                      fontsize=11, fontweight='bold')

ax.set_title('Sequence Power Flow Matrix: d-Orbitals\n|<ψ_m1|ψ_m2>| (Normalized)',
             fontsize=13, fontweight='bold')
ax.set_xlabel('m2 (second sequence)', fontsize=11)
ax.set_ylabel('m1 (first sequence)', fontsize=11)

plt.colorbar(im, ax=ax, label='Coupling Strength')

# Add interpretation
ax.text(1.15, 0.5, 'Diagonal: Self-power\n(always = 1, orthonormal)\n\nOff-diagonal: Cross-power\n(coupling between sequences)\n\nSelection rules:\nm1 = m2 → strong\n|m1 - m2| large → weak',
        transform=ax.transAxes, fontsize=10,
        verticalalignment='center',
        bbox=dict(boxstyle='round', facecolor='lightblue', alpha=0.9))

plt.tight_layout()
plt.show()

print("\n=== Interpretation ===")
print("Diagonal elements: Self-power (normalized to 1)")
print("Off-diagonal elements: Inter-sequence coupling")
print("\nNotice: Matrix is symmetric (Hermitian) - power flow is reciprocal!")
print("Just like sequence power flow in balanced multi-phase systems.")

## Summary: Advanced Multi-Phase Orbital Dynamics

### 1. 5-Phase Systems (d-orbitals)
- Five components (m = -2, -1, 0, +1, +2)
- Includes **2nd harmonic** sequences (m = ±2)
- Richer harmonic content than 3-phase
- Analogous to 5-phase power with phases at 72° increments

### 2. Time-Dependent Superpositions
- Different energy levels → **beating** at Δω = (E₂-E₁)/ℏ
- Degenerate states (same n) → static patterns
- Exactly like:
  - Balanced system: All at same frequency → steady state
  - Unbalanced: Different frequencies → pulsating/beating

### 3. Sequence Power Flow
- Cross-terms ψₘ₁*·ψₘ₂ → inter-sequence power transfer
- Creates **interference patterns** (constructive/destructive)
- Selection rules govern which sequences couple
- Power flow matrix is symmetric (reciprocal)

### Key Insight:
The mathematical structure of orbitals with angular momentum l maps exactly
onto multi-phase power systems with (2l+1) phases:

| l | Orbital | Phases | Harmonics |
|---|---------|--------|------------|
| 0 | s | 1-phase | DC only |
| 1 | p | 3-phase | ±1 (fundamental) |
| 2 | d | 5-phase | ±1, ±2 (fundamental + 2nd) |
| 3 | f | 7-phase | ±1, ±2, ±3 (up to 3rd harmonic) |

### Applications of This Framework:
1. **Understanding orbital hybridization** as multi-phase synchronization
2. **Transition dipole moments** as sequence power flow during state changes
3. **Angular momentum coupling** (adding orbital angular momenta) as combining multi-phase systems
4. **Selection rules** as constraints on which sequences can exchange power

### For Your Three-Phase Work:
Your DSOGI positive/negative sequence separation is **mathematically identical**
to decomposing p-orbitals into m = ±1 components! The Clarke transform in power
systems is the same as transforming between:
- Real basis (px, py, pz) ↔ abc phases
- Complex basis (Y₁⁺¹, Y₁⁰, Y₁⁻¹) ↔ sequence components (α⁺, 0, α⁻)

This is a deep connection rooted in **SO(3) symmetry** - the same mathematics
appears whether you're analyzing:
- Rotations in 3D space (quantum angular momentum)
- Balanced 3-phase electrical systems
- Symmetrical components decomposition

### Next Steps to Explore:
1. Implement Park transform analog for orbitals (rotating reference frame)
2. Study angular momentum addition as "coupling multi-phase systems"
3. Explore Stark effect (external E-field) as unbalanced loading
4. Investigate Zeeman effect (external B-field) as sequence splitting
5. Time-dependent perturbation theory as transient power flow analysis